# 06 · Escolha das features

Ridge com janela expansiva (retreino todo mês). Escolho os grupos de features na validação (2010 a 2014) e só depois olho o teste (2015 a jun/2026).

Duas referências: o **passeio** (mês que vem igual à média deste) e o **último dia** (mês que vem igual ao preço do último dia deste).

In [ ]:
import sys
from pathlib import Path

if Path("/kaggle/input").exists():
    RAIZ = next(Path("/kaggle/input").rglob("src/soja/__init__.py")).parents[2]
else:
    RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from soja import config, avaliacao, experimento

ETAPA = "06_features"
PASTA = config.pasta_resultados(ETAPA)
plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": .3})

tabela = pd.read_csv(config.ler_resultado("05_tabela", "tabela_mensal.csv"), index_col="mes")
tabela.index = pd.PeriodIndex(tabela.index, freq="M")

# a exportação explode na entressafra (quase zero contra o ano anterior): limito em ±1
tabela["exportacao_br_12m"] = tabela["exportacao_br_12m"].clip(-1, 1)

G = config.GRUPOS
colunas = lambda grupos: [c for g in grupos for c in G[g]]
PERIODOS = {"validacao": (config.VALIDACAO_INICIO, config.VALIDACAO_FIM), "teste": (config.TESTE_INICIO, config.TESTE_FIM)}

def referencias(periodo):
    ini, fim = PERIODOS[periodo]
    return {"passeio": pd.Series(0.0, index=tabela.loc[ini:fim].index),
            "ultimo_dia": tabela.loc[ini:fim, "fim_vs_media"]}

def avaliar(nome, grupos, periodo):
    ini, fim = PERIODOS[periodo]
    prev = experimento.janela_expansiva(tabela, colunas(grupos), ini, fim)
    alvo = tabela.loc[prev.index, "alvo"]
    refs = {k: v.loc[prev.index] for k, v in referencias(periodo).items()}
    return prev, {"modelo": nome, "periodo": periodo, "meses": len(prev), **avaliacao.resumo(prev, alvo, refs)}

## 1. Referências

In [ ]:
linhas = []
for periodo, (ini, fim) in PERIODOS.items():
    alvo = tabela.loc[ini:fim, "alvo"]
    ref = referencias(periodo)
    for nome, prev in ref.items():
        linhas.append({"modelo": nome, "periodo": periodo, "meses": len(alvo),
                       **avaliacao.resumo(prev, alvo, {k: v for k, v in ref.items() if k != nome})})
refs = pd.DataFrame(linhas)

pd.DataFrame({p: [avaliacao.rmse(tabela.loc[i:f, "alvo"] - r) for r in referencias(p).values()]
              for p, (i, f) in PERIODOS.items()}, index=["passeio", "último dia"]).T.plot.bar(rot=0, title="RMSE das referências")
plt.show()
refs.round(4)

## 2. Cada grupo somado à base

Base = preço + sazonalidade. Barras abaixo de 1 = melhor que a referência.

In [ ]:
linhas, previsoes = [], {}
candidatos = {"base": config.BASE, **{g: config.BASE + [g] for g in G if g not in config.BASE}}
for nome, grupos in candidatos.items():
    for periodo in PERIODOS:
        prev, r = avaliar(nome, grupos, periodo)
        linhas.append(r)
        previsoes[(nome, periodo)] = prev
por_grupo = pd.DataFrame(linhas)

fig, ax = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for a, ref in zip(ax, ["passeio", "ultimo_dia"]):
    por_grupo.pivot(index="modelo", columns="periodo", values=f"rmse_rel_{ref}")[["validacao", "teste"]].plot.barh(ax=a, title=f"RMSE ÷ {ref}")
    a.axvline(1, color="k", lw=.8)
plt.tight_layout(); plt.savefig(PASTA / "grupos.png", dpi=120); plt.show()
por_grupo.round(3)

## 3. Seleção para frente (só na validação)

A partir da base, adiciono o grupo que mais reduz o erro na validação, um de cada vez, até nenhum ajudar.

In [ ]:
outros = {g: G[g] for g in G if g not in config.BASE}
escolhidos, historico = experimento.selecao_para_frente(tabela, colunas(config.BASE), outros, *PERIODOS["validacao"])
historico = pd.DataFrame(historico, columns=["grupo adicionado", "rmse_validacao"])

historico.set_index("grupo adicionado")["rmse_validacao"].plot(marker="o", title="Erro na validação a cada grupo adicionado")
plt.xticks(range(len(historico)), historico["grupo adicionado"], rotation=30); plt.show()
print("Grupos escolhidos:", config.BASE + escolhidos)
historico.round(4)

## 4. Conjunto escolhido no teste

In [ ]:
final = config.BASE + escolhidos
prev_final, r_final = avaliar("escolhido", final, "teste")
previsoes[("escolhido", "teste")] = prev_final
alvo = tabela.loc[prev_final.index, "alvo"]
ref = referencias("teste")

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
x = prev_final.index.to_timestamp()
ax[0].plot(x, alvo, color="k", lw=.8, label="alvo")
ax[0].plot(x, prev_final, color="tab:red", label="Ridge")
ax[0].plot(x, ref["ultimo_dia"].loc[prev_final.index], color="tab:blue", alpha=.6, label="último dia")
ax[0].legend(); ax[0].set_title("Variação do mês seguinte: real e previsões")
for nome, cor in [("passeio", "tab:gray"), ("ultimo_dia", "tab:blue")]:
    ganho = ((alvo - ref[nome].loc[alvo.index]) ** 2 - (alvo - prev_final) ** 2).cumsum()
    ax[1].plot(x, ganho, color=cor, label=f"contra {nome}")
ax[1].axhline(0, c="k", lw=.7); ax[1].legend(); ax[1].set_title("Vantagem acumulada do Ridge (sobe = Ridge errando menos)")
plt.tight_layout(); plt.savefig(PASTA / "teste.png", dpi=120); plt.show()
pd.Series(r_final).to_frame("escolhido")

## 5. Coeficientes do último treino

In [ ]:
ultimo = experimento.ridge().fit(*(lambda d: (d[colunas(final)], d["alvo"]))(tabela[colunas(final) + ["alvo"]].dropna()))
coef = pd.Series(ultimo[-1].coef_, index=colunas(final)).sort_values()
coef.plot.barh(color=np.where(coef > 0, "tab:green", "tab:red"), figsize=(8, 6),
               title=f"Coeficientes padronizados (alpha = {ultimo[-1].alpha_:.2f})")
plt.axvline(0, c="k", lw=.8); plt.tight_layout(); plt.show()
coef.round(4)

## 6. Salvando

In [ ]:
resumo = pd.concat([refs, por_grupo, pd.DataFrame([r_final])], ignore_index=True)
resumo.to_csv(PASTA / "resumo.csv", index=False)

prev_df = pd.concat({f"{m}|{p}": s for (m, p), s in previsoes.items()}, names=["chave", "mes"]).reset_index()
prev_df[["modelo", "periodo"]] = prev_df["chave"].str.split("|", expand=True)
prev_df["alvo"] = tabela.loc[pd.PeriodIndex(prev_df["mes"]), "alvo"].values
prev_df["mes"] = prev_df["mes"].astype(str)
prev_df[["modelo", "periodo", "mes", "previsao", "alvo"]].to_csv(PASTA / "previsoes.csv", index=False)

colunas_dm = ["modelo", "periodo", "rmse_rel_passeio", "p_passeio", "rmse_rel_ultimo_dia", "p_ultimo_dia"]
resumo.dropna(subset=["p_passeio", "p_ultimo_dia"])[colunas_dm].to_csv(PASTA / "diebold_mariano.csv", index=False)

config.salvar_config(PASTA, modelo="StandardScaler + RidgeCV", alphas=[float(a) for a in experimento.ALPHAS[[0, -1]]],
                     validacao=[config.VALIDACAO_INICIO, config.VALIDACAO_FIM], base=config.BASE,
                     escolhidos=escolhidos, colunas_finais=colunas(final), exportacao_limitada_em=1)
print(sorted(p.name for p in PASTA.iterdir()))